# 00 · Setup check and self-test
### Machine Unlearning is Extinction, Not Forgetting

**Run this notebook first**, and again whenever you change `ext_utils.py`. It spends about 5–10 GPU-minutes
so you don't waste hours of quota on a broken setup. It checks:

1. **Environment**: GPUs, internet, disk space, attached inputs.
2. **Data**: CIFAR-100 can be loaded (from an attached dataset or by download).
3. **Self-test**: every component (training, checkpoints, all 5 unlearning methods, all 4 extinction tests,
   the probe and CKA) runs once on a tiny subset.
4. **Speed benchmark**: one real training epoch, to calibrate the compute budget.

### The whole project

| Notebook | What it does | Needs (attach as input) | Approx. time on T4 x2 |
|---|---|---|---|
| **00** setup check | this notebook | `ext-code` | 10 min |
| **01** originals | trains 3 models on all data | `ext-code` | 0.5 h |
| **02** retrained references | 31 models trained without the forget class ("true forgetting") | `ext-code`, 01 | 3–4 h |
| **03** unlearning | calibration, pilot, 150 unlearned, 75 ABA and 75 multi-context models | `ext-code`, 01, 02 | 6–9 h |
| **04** extinction tests | spontaneous recovery, renewal, reinstatement, savings, plus BatchNorm controls | `ext-code`, 01, 02, 03 (+ CIFAR-100-C) | 8–10 h (2 sessions) |
| **05** mechanism | linear probe, head swap, weight change, CKA | `ext-code`, 01, 02, 03 | 1.5–2.5 h |
| **06** analysis | statistics (H1–H6, BatchNorm controls) and figures, CPU only | outputs of 01–05 | 5 min |
| *optional* GroupNorm arm | notebooks 01–06 again with `norm="gn"`, 1 seed, 5 classes | as above, `-gn` names | 4–6 h |

## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **GPU T4 x2** | Two GPUs: `ext_utils` runs one independent worker per GPU, so two models train at the same time. Your weekly quota is counted in session hours, not per GPU, so T4 x2 roughly doubles what you get per hour. (A single P100 also works: the code detects one GPU and uses one worker.) |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` dataset (contains `ext_utils.py`) | The shared library. Upload it once as a private dataset. |
| **CIFAR-100 Python** dataset ([fedesoriano/cifar100](https://www.kaggle.com/datasets/fedesoriano/cifar100)) | **Strongly recommended.** Without it CIFAR-100 is downloaded from cs.toronto.edu at ~50 kB/s (56 min in run #1). Check that the setup cell prints `CIFAR-100: found at …`. |


**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. For this notebook you can also just run the cells interactively.

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

### One-time step: upload `ext_utils.py` as a dataset

1. On kaggle.com go to **Datasets → New Dataset**.
2. Upload `ext_utils.py`, give it a title such as **ext-code**, and keep it **Private**.
3. In every notebook: **Add Input → Your Datasets → ext-code**.

**When you change `ext_utils.py` later:** open the dataset → **New Version** → upload the new file. In each
notebook, update the input to the new version (the input panel shows a refresh icon), then restart the session.

*Why a dataset instead of pasting the code into every notebook?* There is then exactly one copy of the code,
so the notebooks can never drift apart. Every run also saves the exact version it used as
`ext_utils_used.py` in its output, as a record of what ran.

To share with your co-author: open the dataset → **Settings → Sharing** → add their Kaggle username. They
attach it the same way.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Self-test on a tiny subset

`U.self_test()` builds a *smoke* configuration (30 training and 10 test images per class, 1 epoch, a narrow
width-16 network) and runs every component once. Each line prints **PASS** or **FAIL** with the full
error. It writes nothing permanent.

| What is checked | Why it matters |
|---|---|
| training, checkpoint save/load, metrics | notebooks 01 and 02 |
| the 5 input contexts (clean, gray, blur, noise, contrast) | renewal tests and ABA unlearning |
| FT, NegGrad+, RL, SCRUB, SalUn, multi-context unlearning | notebook 03 |
| BatchNorm recalibration control, GroupNorm model | BatchNorm controls (notebook 04, GroupNorm arm) |
| spontaneous recovery (BN updating and frozen), reinstatement, savings | notebook 04 |
| linear probe, CKA, weight change | notebook 05 |

**All lines should say PASS.** If one fails, fix it before running anything else. The traceback shows the cause.

In [ ]:
results = U.self_test()
failed = {k: v for k, v in results.items() if v != "PASS"}
assert not failed, f"Self-test failures: {failed}"
print("\nAll components work. You can start notebook 01.")

## Check the real configuration and data

`U.make_cfg()` returns the full configuration: the defaults in `ext_utils.DEFAULTS`, plus anything you pass
in. A misspelled parameter name raises an error instead of being silently ignored.

The cell below loads the **full** CIFAR-100 set and checks that all forget classes and the pilot class
exist. It also prints each class's superclass siblings, which are the "related" classes for the
reinstatement test and the collateral-damage analysis.

In [ ]:
CFG = U.make_cfg(smoke=False)
D = U.load_data(CFG)
U.validate_classes(CFG, D)
print(f"train images: {len(D['ytr'])}, test images: {len(D['yte'])}")
for name in U.all_classes(CFG):
    c = U.cid(D, name)
    sup = D["coarse_names"][D["f2c"][c]]
    sibs = [D["fine_names"][k] for k in U.siblings(D, c)]
    print(f"{name:12s} superclass={sup:32s} siblings={sibs}")

## Speed benchmark: one real training epoch

This trains the real ResNet-18 (width 64, batch 256, mixed precision) for **one epoch on one GPU** and
projects the compute budget from it. Expect roughly 15–30 s per epoch on a T4 (the first epoch includes
cuDNN warm-up).

Set `RUN_BENCHMARK = False` to skip it.

In [ ]:
RUN_BENCHMARK = True

if RUN_BENCHMARK:
    import time, torch
    device = U.get_device()
    torch.backends.cudnn.benchmark = CFG["cudnn_benchmark"]
    bench_cfg = dict(CFG, epochs=1)
    G = U.gpu_data(D, bench_cfg, device)
    t = time.time()
    U.train_model(bench_cfg, G, G["all_idx"], seed=0, device=device, tag="benchmark")
    sec = time.time() - t
    per_model_h = sec * CFG["epochs"] / 3600
    n_models = len(CFG["seeds"]) + len(U.jobs_retrain(CFG))
    n_gpu = max(1, torch.cuda.device_count())
    print(f"\n{sec:.1f} s/epoch -> {per_model_h * 60:.1f} min per {CFG['epochs']}-epoch model")
    print(f"notebooks 01+02 ({n_models} models): ~{n_models * per_model_h:.1f} GPU-hours "
          f"= ~{n_models * per_model_h / n_gpu:.1f} h wall time with {n_gpu} GPU(s)")

## Next step

If everything passed, open **notebook 01**. You don't need to save a version of this notebook.

**Optional dry run of the whole chain.** Set `smoke=True` in notebooks 01–06 and run them in order (about
30–60 minutes in total). This checks the hand-over of files between notebooks before you spend real quota.
Smoke files are named `*_smoke*` and are ignored by real runs.

## Name of this run

Title this notebook **`ext-00-setup-selftest`**. Its output is not needed by any other notebook, so no dataset is required. The code itself lives in the private dataset **`ext-code`**.